In [ ]:
import os
print(os.listdir())


In [ ]:
import os
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())


In [ ]:
import zipfile
import pandas as pd

with zipfile.ZipFile("gtfs.zip", 'r') as zip_ref:
    zip_ref.extractall("gtfs_nairobi/")

stops = pd.read_csv("gtfs_nairobi/stops.txt")
stops.head()


In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Select only lat/lon columns and drop NaNs if any
coords = stops[['stop_lat', 'stop_lon']].dropna()

# Try with k=5 clusters (you can tweak later)
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(coords)

# Add cluster labels back to dataframe
stops['cluster'] = kmeans.labels_

# Plot clusters
plt.figure(figsize=(10, 8))
plt.scatter(stops['stop_lon'], stops['stop_lat'], c=stops['cluster'], cmap='tab10', s=50)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Nairobi Public Transport Stops Clustered')
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)


In [ ]:
print(stops['cluster'].value_counts())


In [ ]:
!pip install folium


In [ ]:
import sys
!{sys.executable} -m pip install folium


In [ ]:
import sys
print(sys.executable)


In [ ]:
!/opt/conda/envs/anaconda-2024.02-py310/bin/python -m pip install folium


In [ ]:
import folium
from folium.plugins import MarkerCluster


In [ ]:
kmeans = KMeans(n_clusters=5, random_state=0, n_init='auto')


In [ ]:
import zipfile
import pandas as pd

# Unzip GTFS file if needed
with zipfile.ZipFile("gtfs.zip", 'r') as zip_ref:
    zip_ref.extractall("gtfs_nairobi/")

# Load stops data
stops = pd.read_csv("gtfs_nairobi/stops.txt")

# Do clustering again
from sklearn.cluster import KMeans

X = stops[['stop_lat', 'stop_lon']]
kmeans = KMeans(n_clusters=5, random_state=0)
stops['cluster'] = kmeans.fit_predict(X)


In [ ]:
kmeans = KMeans(n_clusters=5, random_state=0, n_init='auto')


In [ ]:
from geopy.distance import geodesic
import folium

# Simple nearest neighbor algorithm
def nearest_neighbor_route(stops_df):
    unvisited = stops_df.copy()
    route = [unvisited.iloc[0]]
    unvisited = unvisited.drop(unvisited.index[0])

    while not unvisited.empty:
        last_stop = route[-1]
        distances = unvisited.apply(
            lambda row: geodesic(
                (last_stop['stop_lat'], last_stop['stop_lon']),
                (row['stop_lat'], row['stop_lon'])
            ).meters,
            axis=1
        )
        nearest_index = distances.idxmin()
        route.append(unvisited.loc[nearest_index])
        unvisited = unvisited.drop(index=nearest_index)

    return pd.DataFrame(route)

# Let's pick one cluster to simulate route optimization — for example cluster 1
cluster_id = 1
cluster_stops = stops[stops['cluster'] == cluster_id].reset_index(drop=True)

# Get optimized route
optimized_route_df = nearest_neighbor_route(cluster_stops)

# Visualize the optimized route on a map
route_map = folium.Map(location=[optimized_route_df['stop_lat'].mean(), optimized_route_df['stop_lon'].mean()], zoom_start=12)

# Add the route
for i in range(len(optimized_route_df) - 1):
    point1 = (optimized_route_df.iloc[i]['stop_lat'], optimized_route_df.iloc[i]['stop_lon'])
    point2 = (optimized_route_df.iloc[i+1]['stop_lat'], optimized_route_df.iloc[i+1]['stop_lon'])
    folium.PolyLine([point1, point2], color="blue", weight=2.5, opacity=1).add_to(route_map)

# Add stop markers
for _, row in optimized_route_df.iterrows():
    folium.Marker(
        location=[row['stop_lat'], row['stop_lon']],
        popup=row['stop_name'],
        icon=folium.Icon(color='green', icon='bus', prefix='fa')
    ).add_to(route_map)

route_map
